# 01 — Extracción, Infraestructura y Auditoría

Este notebook corresponde a la **Persona 1**. Descarga los Parquet, crea la estructura `raw`, lee cada archivo individualmente con Spark y genera `audit_file_inventory`.

In [1]:
%pip install -q pyspark pyyaml requests
%pip install -q pyarrow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys
from pathlib import Path

# Ubicación del proyecto. En Colab puedes ajustar esta ruta si subes la carpeta a Drive.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

print("Proyecto:", PROJECT_ROOT)

Proyecto: c:\Modelado\ProyectoSegundoParcial


In [3]:
from utils import load_config, ensure_directories, generate_process_id, get_spark_session
from extract import download_nyc_files, download_apache_bad_data, build_audit_file_inventory, write_audit_inventory

config = load_config("config/etl_config.yaml")
ensure_directories(config)
process_id = generate_process_id()

print("process_id:", process_id)

process_id: dabbb348-001b-4035-ae4b-0617016818ff


## 1. Descarga de archivos NYC TLC

Los archivos se almacenan sin modificación en `data/raw/` respetando particiones `year=` y `month=`.

In [4]:
# Esta celda puede tardar por el tamaño de los archivos.
download_results = download_nyc_files(config, overwrite=False)
download_results[:3], len(download_results)

([{'file_name': 'yellow_tripdata_2023-01.parquet',
   'service_type': 'yellow',
   'year': 2023,
   'month': 1,
   'url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet',
   'target_path': 'C:\\Modelado\\ProyectoSegundoParcial\\data\\raw\\yellow\\year=2023\\month=01\\yellow_tripdata_2023-01.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': '32df6f67578fa86c484a6b5ef23a5281992ff085521082340b0f9e5889e9a572'},
  {'file_name': 'yellow_tripdata_2023-02.parquet',
   'service_type': 'yellow',
   'year': 2023,
   'month': 2,
   'url': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet',
   'target_path': 'C:\\Modelado\\ProyectoSegundoParcial\\data\\raw\\yellow\\year=2023\\month=02\\yellow_tripdata_2023-02.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': '4809e6aaac64f05a62d16a25d55713be1537ad64fc261e895eaf2d2120fe750a'},
  {'file_name': 'yellow_tripdata_2023-03.parquet',
   'service_type': 'yel

## 2. Descarga de archivos problemáticos Apache Parquet Testing

Se descargan archivos desde la carpeta `bad_data` para probar errores de lectura y aislamiento de corruptos.

In [5]:
bad_results = download_apache_bad_data(config, overwrite=False)
bad_results[:5], len(bad_results)

([{'file_name': 'ARROW-GH-41317.parquet',
   'url': 'https://raw.githubusercontent.com/apache/parquet-testing/master/bad_data/ARROW-GH-41317.parquet',
   'target_path': 'C:\\Modelado\\ProyectoSegundoParcial\\data\\raw\\bad_parquet\\ARROW-GH-41317.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': 'a9c2b53992b6c017d3cbe44b9d3a0b2ebbb0f68c8eda424f11942d3313bd5696'},
  {'file_name': 'ARROW-GH-41321.parquet',
   'url': 'https://raw.githubusercontent.com/apache/parquet-testing/master/bad_data/ARROW-GH-41321.parquet',
   'target_path': 'C:\\Modelado\\ProyectoSegundoParcial\\data\\raw\\bad_parquet\\ARROW-GH-41321.parquet',
   'download_status': 'SKIPPED_EXISTS',
   'sha256_file': '9539e0925ddb0b52a467cfe5535455be26869bb9bf3b5e48a255c7e2c1973537'},
  {'file_name': 'ARROW-GH-43605.parquet',
   'url': 'https://raw.githubusercontent.com/apache/parquet-testing/master/bad_data/ARROW-GH-43605.parquet',
   'target_path': 'C:\\Modelado\\ProyectoSegundoParcial\\data\\raw\\bad_parquet\\

## 3. Inicialización de Spark

In [6]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Python usado por Spark:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

Python usado por Spark: c:\Users\HP OMEN\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


In [7]:
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")
spark

## 4. Lectura segura individual y generación de inventario

Cada archivo se lee dentro de `try/except`. Si falla, el error queda en `error_message` y el pipeline continúa.

In [8]:
inventory_df = build_audit_file_inventory(spark, config, process_id)

# Validación de columnas exactas
expected_cols = [
    "process_id", "source_system", "service_type", "file_name", "file_path",
    "file_size_mb", "file_hash_sha256", "partition_year", "partition_month", "read_status",
    "record_count", "column_count", "schema_hash", "error_message", "processed_at"
]
assert inventory_df.columns == expected_cols, inventory_df.columns

inventory_df.show(50, truncate=80)


+------------------------------------+----------------------+------------+-----------------------------------+--------------------------------------------------------------------------------+------------+----------------------------------------------------------------+--------------+---------------+------------+------------+------------+----------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------+
|                          process_id|         source_system|service_type|                          file_name|                                                                       file_path|file_size_mb|                                                file_hash_sha256|partition_year|partition_month| read_status|record_count|column_count|                                                     schema_hash|                                                                   error_message|  

In [9]:
audit_path = write_audit_inventory(inventory_df, config)
print("audit_file_inventory escrito en:", audit_path)

audit_file_inventory escrito en: C:\Modelado\ProyectoSegundoParcial\data\audit\audit_file_inventory


## 5. Resumen de estados de lectura

In [10]:
inventory_df.groupBy("source_system", "service_type", "read_status").count().orderBy("source_system", "service_type", "read_status").show(truncate=False)

+----------------------+------------+------------+-----+
|source_system         |service_type|read_status |count|
+----------------------+------------+------------+-----+
|APACHE_PARQUET_TESTING|bad_parquet |CORRUPT     |1    |
|APACHE_PARQUET_TESTING|bad_parquet |SCHEMA_ERROR|1    |
|APACHE_PARQUET_TESTING|bad_parquet |SUCCESS     |7    |
|NYC_TLC               |fhvhv       |SUCCESS     |1    |
|NYC_TLC               |green       |SUCCESS     |2    |
|NYC_TLC               |yellow      |SUCCESS     |8    |
+----------------------+------------+------------+-----+



## 6. Validación de archivos corruptos, vacíos y exitosos

In [11]:
from pyspark.sql.functions import col

print("Archivos corruptos o con error:")
inventory_df.filter(col("read_status").isin("CORRUPT", "SCHEMA_ERROR")).select(
    "service_type", "file_name", "read_status", "error_message"
).show(30, truncate=100)

print("Archivos vacíos:")
inventory_df.filter(col("read_status") == "EMPTY").select(
    "service_type", "file_name", "record_count"
).show(30, truncate=False)

print("Archivos leídos correctamente:")
inventory_df.filter(col("read_status") == "SUCCESS").select(
    "service_type", "file_name", "record_count", "column_count", "schema_hash"
).show(30, truncate=80)

Archivos corruptos o con error:
+------------+--------------------+------------+----------------------------------------------------------------------------------------------------+
|service_type|           file_name| read_status|                                                                                       error_message|
+------------+--------------------+------------+----------------------------------------------------------------------------------------------------+
| bad_parquet|PARQUET-1481.parquet|SCHEMA_ERROR|                                                               Invalid physical column type: UNKNOWN|
| bad_parquet|           README.md|     CORRUPT|Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.|
+------------+--------------------+------------+----------------------------------------------------------------------------------------------------+

Archivos vacíos:
+------------+---------+------------+
|service_typ

## 7. Lectura de carpetas particionadas completas

Esta sección demuestra que Spark puede leer carpetas por tipo de servicio usando particiones `year/month`.

In [12]:
import pandas as pd
from pathlib import Path
from utils import resolve_path

for service in ["yellow", "green", "fhvhv"]:
    try:
        service_path = resolve_path(config, "raw") / service
        parquet_files = list(service_path.rglob("*.parquet"))
        if not parquet_files:
            print(f"{service}: No se encontraron archivos")
            continue
        df_pd = pd.read_parquet(str(parquet_files[0]), engine="pyarrow")
        print(f"Servicio: {service}")
        print(f"Columnas: {len(df_pd.columns)}")
        for col_name, dtype in df_pd.dtypes.items():
            print(f"  |-- {col_name}: {dtype}")
        print()
    except Exception as e:
        print(f"No se pudo leer {service}: {e}")


Servicio: yellow
Columnas: 19
  |-- VendorID: int64
  |-- tpep_pickup_datetime: datetime64[us]
  |-- tpep_dropoff_datetime: datetime64[us]
  |-- passenger_count: float64
  |-- trip_distance: float64
  |-- RatecodeID: float64
  |-- store_and_fwd_flag: str
  |-- PULocationID: int64
  |-- DOLocationID: int64
  |-- payment_type: int64
  |-- fare_amount: float64
  |-- extra: float64
  |-- mta_tax: float64
  |-- tip_amount: float64
  |-- tolls_amount: float64
  |-- improvement_surcharge: float64
  |-- total_amount: float64
  |-- congestion_surcharge: float64
  |-- airport_fee: float64

Servicio: green
Columnas: 20
  |-- VendorID: int64
  |-- lpep_pickup_datetime: datetime64[us]
  |-- lpep_dropoff_datetime: datetime64[us]
  |-- store_and_fwd_flag: str
  |-- RatecodeID: float64
  |-- PULocationID: int64
  |-- DOLocationID: int64
  |-- passenger_count: float64
  |-- trip_distance: float64
  |-- fare_amount: float64
  |-- extra: float64
  |-- mta_tax: float64
  |-- tip_amount: float64
  |-- toll

## 8. Evidencia de idempotencia

La tabla `audit_file_inventory` se escribe en modo `overwrite`, por eso una segunda ejecución reemplaza el resultado anterior y no duplica registros.

In [13]:
import pandas as pd

audit_check = pd.read_parquet(audit_path, engine="pyarrow")
print("Registros en audit_file_inventory:", len(audit_check))
print()
print(audit_check.groupby("process_id").size().reset_index(name="count").to_string(index=False))


Registros en audit_file_inventory: 20

                          process_id  count
dabbb348-001b-4035-ae4b-0617016818ff     20
